In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

uniprot_path = ROOT / "data/raw/uniprot/uniprotkb_organism_id_9606_AND_reviewed_2026_10_12.tsv.gz"

uniprot_df = pd.read_csv(
    uniprot_path,
    sep="\t",
    compression="gzip",
    dtype=str
)

In [2]:
mapped_df = uniprot_df[
    uniprot_df["STRING"].notna()
    & uniprot_df["STRING"].str.strip().ne("")
].copy()

mapped_df["string_id_list"] = mapped_df["STRING"].apply(
    lambda value: [item.strip() for item in value.split(";") if item.strip()]
)

mapped_df["string_id_count"] = mapped_df["string_id_list"].str.len()

print("Proteins with at least one STRING ID:", len(mapped_df))
print("\nSTRING IDs per UniProt protein:")
print(mapped_df["string_id_count"].value_counts().sort_index())

Proteins with at least one STRING ID: 18841

STRING IDs per UniProt protein:
string_id_count
1    18841
Name: count, dtype: int64


In [3]:
mapped_df[
    [
        "Entry",
        "Gene Names (primary)",
        "Protein names",
        "STRING",
        "string_id_count",
    ]
].head(10)

,Entry,Gene Names (primary),Protein names,STRING,string_id_count
1,A0A096LP01,SMIM26,Small integral membrane protein 26,9606.ENSP00000485316;,1
2,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),9606.ENSP00000484893;,1
5,A0A0U1RRE5,NBDY,Negative regulator of P-body association (P-bo...,9606.ENSP00000489583;,1
9,A0AV02,SLC12A8,Solute carrier family 12 member 8 (Cation-chlo...,9606.ENSP00000377112;,1
10,A0AV96,RBM47,RNA-binding protein 47 (RNA-binding motif prot...,9606.ENSP00000295971;,1
11,A0AVF1,IFT56,Intraflagellar transport protein 56 (Tetratric...,9606.ENSP00000419279;,1
12,A0AVI4,TMEM129,E3 ubiquitin-protein ligase TM129 (EC 2.3.2.27...,9606.ENSP00000372394;,1
13,A0AVK6,E2F8,Transcription factor E2F8 (E2F-8),9606.ENSP00000250024;,1
14,A0AVT1,UBA6,Ubiquitin-like modifier-activating enzyme 6 (U...,9606.ENSP00000313454;,1
15,A0FGR8,ESYT2,Extended synaptotagmin-2 (E-Syt2) (Chr2Syt),9606.ENSP00000499020;,1


In [5]:
# STRING ID ko clean format mein nikalo: trailing ";" hata do
mapped_df["string_id"] = (
    mapped_df["STRING"]
    .str.strip()
    .str.rstrip(";")
)

print("Total mapped UniProt proteins :", len(mapped_df))
print("Unique UniProt IDs            :", mapped_df["Entry"].nunique())
print("Unique STRING IDs             :", mapped_df["string_id"].nunique())

duplicate_string_rows = mapped_df[
    mapped_df["string_id"].duplicated(keep=False)
].sort_values("string_id")

print("\nRows involved in duplicated STRING mappings:", len(duplicate_string_rows))

if len(duplicate_string_rows) > 0:
    display(
        duplicate_string_rows[
            ["Entry", "Gene Names (primary)", "string_id"]
        ].head(20)
    )
else:
    print("Perfect one-to-one mapping confirmed.")

Total mapped UniProt proteins : 18841
Unique UniProt IDs            : 18841
Unique STRING IDs             : 18841

Rows involved in duplicated STRING mappings: 0
Perfect one-to-one mapping confirmed.
